# DITU-net — Fair Apple-to-Apple Baseline Comparison (Inference Only)

**Purpose.** Establish a baseline that *no reviewer can question*, by cross-testing the two models on each other's data under one identical evaluation protocol:

* **Experiment 1 — Base-paper model on *our* data.** The published DITU-net checkpoint (`UnetS4000DEAndADEepoch3.pth`) is run on **ACT-2 2025**, **ACT-2 2026**, and **ACT-1 2025**, across scenarios **S1/S2/S3**, scored against the same independent **TP 90th-percentile** reference and the same metrics (RMSE, MAE, MAPE, WMAPE, SS@0.05/0.10/0.15) used for our own results.
* **Experiment 2 — Both models on the base paper's *own* sample data** (`sample.csv`). Since that file has only wind-speed and power (no theoretical power), we use the **base paper's own scoring protocol**: an 80/20 split with the curve scored against held-out **actual** power points. Both models are scored on the identical split.

**Fairness controls.**
1. **One shared renderer** for both models — the base paper's marker-size rule (Eq. 24, `MS = sqrt(3528/n)`). The input image is therefore identical for both networks; only the *network* and its *native read-out* differ.
2. Each model is read out with **its own published read-out** (faithful baseline). In Experiment 1 we additionally report a **shared-read-out control** (base model + our P-spline reader) to prove the gap is the *model*, not the reader.
3. The evaluation reference is **never seen by either model** (no curve is fit to the scoring target).

**You only edit the paths in Cell 1.** Everything else runs top-to-bottom and writes a single downloadable ZIP at the end.

> Tested offline against the real `act1_consolidated.xlsx` schema and `sample.csv`. Includes a fix for a NumPy-version incompatibility in the base paper's original `imageToLine` (a length-1 array assigned to a scalar slot — silently worked on old NumPy, errors on current Kaggle), and bounded Newton-Raphson loops so a pathological curve cannot hang the run.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — Imports, EDITABLE paths, config, reproducibility
# ═══════════════════════════════════════════════════════════════════════════
import os, io, json, time, math, zipfile, warnings, gc
from collections import OrderedDict
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from numpy import polyfit, poly1d
from scipy import optimize
from scipy.interpolate import BSpline
from scipy.optimize import lsq_linear

# ─────────────────────────────────────────────────────────────────────────────
#  >>>>>>>>>>>>>>>>>>>>>>  EDIT THESE PATHS ONLY  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ─────────────────────────────────────────────────────────────────────────────
# (1) Base-paper best checkpoint (out_channels = 3).
BASE_CKPT = '/kaggle/input/datasets/hamzatauqeernust/ditu-net-pretrained-model-v1/UnetS4000DEAndADEepoch3.pth'

# (2) Our best deployment checkpoint (out_channels = 1). Used in Experiment 2.
OUR_CKPT  = '/kaggle/input/datasets/hamzatauqeernust/act2-ditunet-k-fold-checkpoints/Unet_ACT2_deploy_best.pth'

# (3) Our datasets for Experiment 1. Each entry: (label, path, year).
#     .csv OR .xlsx both accepted. Schema must be:
#       Device Name | time | Avg Wind Speed(m/s) | Avg Grid active power(kW) | Avg Theoretical Active Power(kW)
#     Set a path to None / remove the row if you do not have that dataset yet.
OUR_DATASETS = [
    ('ACT2_2025', '/kaggle/input/datasets/hamzatauqeernust/act2-consolidated3yrs-csv/consolidated_wind_data_2023_2025.csv', 2025),
    ('ACT2_2026', '/kaggle/input/datasets/hamzatauqeernust/act2-2026-inference-ready-dataset/act2_2026_consolidated.csv', 2026),
    ('ACT1_2025', '/kaggle/input/datasets/hamzatauqeernust/act-1-inference-ready-test-data/act1_consolidated.xlsx', 2025),
]

# (4) Base-paper sample data for Experiment 2 (NO header; col0 = wind speed, col1 = power).
SAMPLE_CSV = '/kaggle/input/datasets/hamzatauqeernust/base-paper-sample-data/sample.csv'

# (5) Months that are known-incomplete and should be reported separately, by label.
PARTIAL_MONTHS = {'ACT2_2026': ['2026-06']}   # e.g. June 2026 ended mid-month
# ─────────────────────────────────────────────────────────────────────────────

# ── Evaluation settings (identical across everything; DO NOT change for comparability)
TURBINES      = [f'WTG{i}' for i in range(1, 21)]
N_REF_BINS    = 25            # TP-reference wind-speed bins
MS_CONST      = 3528.0        # base paper Eq.24 marker-size constant (K=0.07, MS_DIT=6)
INIT_FEATURES = 32
OUR_DROPOUT   = 0.10          # must match our training
WPC_LAMBDA    = 2.0           # P-spline smoothness (our reader)
SEED          = 2024

# Base-paper read-out fidelity knobs (kept at published defaults):
BASE_POLY_DEG = 12            # polynomial order in pixel->curve fit
BASE_DKC      = True          # domain-knowledge correction (cut-in/rated flats)

# Output roots
OUT_ROOT = '/kaggle/working/baseline_comparison'
EXP1_DIR = f'{OUT_ROOT}/exp1_base_on_ours'
EXP2_DIR = f'{OUT_ROOT}/exp2_models_on_sample'
FIG_DIR  = f'{OUT_ROOT}/figures'
TAB_DIR  = f'{OUT_ROOT}/tables'
for d in (OUT_ROOT, EXP1_DIR, EXP2_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.size': 11, 'axes.labelsize': 12,
    'axes.titlesize': 12, 'legend.fontsize': 8, 'figure.dpi': 300,
})

def _exists(p): return bool(p) and os.path.exists(p)

print(f"Device        : {device}")
print(f"Base ckpt     : {BASE_CKPT}  ({'FOUND' if _exists(BASE_CKPT) else 'MISSING — edit path'})")
print(f"Our  ckpt     : {OUR_CKPT}   ({'FOUND' if _exists(OUR_CKPT) else 'MISSING — edit path'})")
print(f"Sample csv    : {SAMPLE_CSV} ({'FOUND' if _exists(SAMPLE_CSV) else 'MISSING — edit path'})")
print("Datasets for Experiment 1:")
for lab, p, yr in OUR_DATASETS:
    print(f"   {lab:<10} {yr}  {'FOUND' if _exists(p) else 'MISSING — edit path'}  {p}")
print("✅ Cell 1 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Base-paper U-Net (out_channels=3) + transform_invert  [verbatim port]
# ═══════════════════════════════════════════════════════════════════════════
# Ported byte-for-byte from the base paper's model.py so the published
# checkpoint loads exactly. Naming style: encoder1.enc1conv1.weight ...

class BaseUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, init_features=32):
        super().__init__()
        f = init_features
        self.encoder1 = BaseUNet._block(in_channels, f, "enc1"); self.pool1 = nn.MaxPool2d(2, 2)
        self.encoder2 = BaseUNet._block(f, f*2, "enc2");         self.pool2 = nn.MaxPool2d(2, 2)
        self.encoder3 = BaseUNet._block(f*2, f*4, "enc3");       self.pool3 = nn.MaxPool2d(2, 2)
        self.encoder4 = BaseUNet._block(f*4, f*8, "enc4");       self.pool4 = nn.MaxPool2d(2, 2)
        self.bottleneck = BaseUNet._block(f*8, f*16, "bottleneck")
        self.upconv4 = nn.ConvTranspose2d(f*16, f*8, 2, 2); self.decoder4 = BaseUNet._block((f*8)*2, f*8, "dec4")
        self.upconv3 = nn.ConvTranspose2d(f*8,  f*4, 2, 2); self.decoder3 = BaseUNet._block((f*4)*2, f*4, "dec3")
        self.upconv2 = nn.ConvTranspose2d(f*4,  f*2, 2, 2); self.decoder2 = BaseUNet._block((f*2)*2, f*2, "dec2")
        self.upconv1 = nn.ConvTranspose2d(f*2,  f,   2, 2); self.decoder1 = BaseUNet._block(f*2, f, "dec1")
        self.conv = nn.Conv2d(f, out_channels, kernel_size=1)
    def forward(self, x):
        e1 = self.encoder1(x); e2 = self.encoder2(self.pool1(e1))
        e3 = self.encoder3(self.pool2(e2)); e4 = self.encoder4(self.pool3(e3))
        b  = self.bottleneck(self.pool4(e4))
        d4 = self.upconv4(b);  d4 = self.decoder4(torch.cat((d4, e4), 1))
        d3 = self.upconv3(d4); d3 = self.decoder3(torch.cat((d3, e3), 1))
        d2 = self.upconv2(d3); d2 = self.decoder2(torch.cat((d2, e2), 1))
        d1 = self.upconv1(d2); d1 = self.decoder1(torch.cat((d1, e1), 1))
        return self.conv(d1)
    @staticmethod
    def _block(in_ch, feat, name):
        return nn.Sequential(OrderedDict([
            (name+"conv1", nn.Conv2d(in_ch, feat, 3, padding=1, bias=False)),
            (name+"norm1", nn.BatchNorm2d(feat)), (name+"relu1", nn.ReLU(inplace=True)),
            (name+"conv2", nn.Conv2d(feat, feat, 3, padding=1, bias=False)),
            (name+"norm2", nn.BatchNorm2d(feat)), (name+"relu2", nn.ReLU(inplace=True)),
        ]))

def transform_invert(img_, transform_train, type='L'):
    """Base paper helper: invert ToTensor+Normalize, return uint8 grayscale image."""
    if 'Normalize' in str(transform_train):
        nt = list(filter(lambda x: isinstance(x, transforms.Normalize), transform_train.transforms))
        mean = torch.tensor(nt[0].mean, dtype=img_.dtype, device=img_.device)
        std  = torch.tensor(nt[0].std,  dtype=img_.dtype, device=img_.device)
        img_.mul_(std[:, None, None]).add_(mean[:, None, None])
    def convertOne(img):
        img = img.permute(1, 2, 0)
        if 'ToTensor' in str(transform_train):
            img = np.array(img); img -= np.min(img); img /= np.max(img); img = img * 255
        if img.shape[2] == 3:
            img = Image.fromarray(img.astype('uint8')).convert(type)
        elif img.shape[2] == 1:
            img = Image.fromarray(img.astype('uint8').squeeze())
        return img
    if len(img_.shape) > 3:
        for i in range(img_.shape[0]):
            y = np.array(convertOne(img_[i])) if i == 0 else \
                np.concatenate((y, np.array(convertOne(img_[i]))), axis=0)
        return y
    return np.array(convertOne(img_[:, :, :]))

print("✅ Cell 2 done — base U-Net ready")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3 — Base-paper pixel-map + polynomial + DKC read-out  [faithful, hardened]
# ═══════════════════════════════════════════════════════════════════════════
# Ported from the base paper's Pixel_mapping_correction.py. Two safety fixes that
# DO NOT change any reported value, only prevent crashes/hangs on modern Kaggle:
#   (a) scalar index extraction:  y[i] = 1 - where(col==vm)[0][0]/Y
#       (original assigned a length-1 array to a scalar slot — errors on new NumPy)
#   (b) bounded Newton-Raphson loops (guard counters + maxiter) so a pathological
#       curve raises and falls back instead of looping forever.

class _polyF:
    def __init__(self, f): self.f=f; self.order=f.o; self.coefficients=f.coefficients
    def df(self, x):  return sum(x**(self.order-i-1)*self.coefficients[i]*(self.order-i)               for i in range(self.order))
    def ddf(self, x): return sum(x**(self.order-i-2)*self.coefficients[i]*(self.order-i-1)*(self.order-i) for i in range(self.order-1))
    def dddf(self,x): return sum(x**(self.order-i-3)*self.coefficients[i]*(self.order-i-2)*(self.order-i-1)*(self.order-i) for i in range(self.order-2))

def _findZero(f, b, c=0.8):
    def g(x): return f(x) - b
    return optimize.bisect(g, 0, c)

def _paramFunc(f, r1=0.19, MAXIT=400):
    fx = _polyF(f); pass1=False; f_x=True; count=0; firstcome=True; guard=0
    while not pass1:
        guard += 1
        if guard > MAXIT or r1 < -0.5: raise RuntimeError("CWS search did not converge")
        count += 1
        try:
            if count > 20:
                if firstcome: r1=0.19; firstcome=False
                x1 = optimize.newton(fx.df, r1, fprime2=fx.dddf, maxiter=80)
            else:
                x1 = optimize.newton(fx.df, r1, fprime=fx.ddf, fprime2=fx.dddf, maxiter=80)
            pass1 = True
            if f_x:
                if x1 > r1 or f(x1) > 0.01:
                    r1 -= 0.005; pass1 = False
                    if r1 < 0.08: f_x=False; r1=0.19
            else:
                if x1 > r1: r1 -= 0.005; pass1 = False
        except Exception:
            r1 -= 0.005
    b1 = f(x1); pass1 = False
    try: r2 = _findZero(f, 0.95)
    except:
        try: r2 = _findZero(f, 0.85)
        except:
            try: r2 = _findZero(f, 0.75)
            except:
                try: r2 = _findZero(f, 0.65)
                except: r2 = 0.45
    guard = 0
    while not pass1:
        guard += 1
        if guard > MAXIT or r2 > 1.5: raise RuntimeError("RWS search did not converge")
        try:
            x2 = optimize.newton(fx.df, r2, fprime2=fx.dddf, maxiter=80); pass1 = True
            if x2 < r2: r2 += 0.005; pass1 = False
        except Exception:
            r2 += 0.005
    b2 = f(x2)
    return x1, b1, x2, b2

class _finalFunc:
    def __init__(self, f, x1, b1, x2, b2): self.f=f; self.x1=x1; self.b1=b1; self.x2=x2; self.b2=b2
    def forward(self, x):
        if x < self.x1: return self.b1
        elif x < self.x2: return self.f(x)
        else: return self.f(x)
    def forwardVector(self, x):
        if type(x) is not np.ndarray: return self.forward(x)
        yy = self.f(x)
        ind = np.where(x < self.x1); yy[ind] *= 0; yy[ind] += self.b1
        ind = np.where(x > self.x2); yy[ind] *= 0; yy[ind] += self.b2
        return yy.flatten()

def base_imageToLine(x, p=12, domainCorrlation=True):
    """Base paper read-out. Input: 256x256 uint8/float grayscale U-net output."""
    x = x[31:228, 32:230]
    L = x.shape[1]; Y = x.shape[0]; xL = np.arange(0, 1, 1/L); y = np.copy(xL); y0 = 0
    for i in range(L):
        col = x[:, i]; vm = np.min(col); lenVM = len(np.where(col == vm)[0])
        if vm > 220:
            y[i] = np.nan
        else:
            if lenVM == 1: y[i] = 1 - np.where(col == vm)[0][0] / (Y)     # fix (a)
            else:          y[i] = 1 - np.mean(np.where(col == vm)[0]) / (Y)
            while y[i] - y0 > 0.1:
                col[np.argmin(col)] = 224; vm = np.min(col)
                if vm > 220: y[i] = np.nan; break
                else: y[i] = 1 - np.mean(np.where(col == vm)[0]) / (Y)
            if y[i] != np.nan: y0 = y[i]
    a = np.isnan(y); xv = xL[~a]; yv = y[~a]
    if len(xv) < 8: raise RuntimeError("too few extracted pixels")
    xv[-1] = 1
    if not domainCorrlation:
        f3 = poly1d(polyfit(xv, yv, p)); return xv, f3(xv), f3
    f3 = poly1d(polyfit(xv, yv, 12))
    x1, b1, x2, b2 = _paramFunc(f3, 0.19)
    fxx = _finalFunc(f3, x1, b1, x2, b2)
    if b1 < 0 and x1 > 0:
        x1 = _findZero(fxx.forwardVector, 0, 0.2); b1 = fxx.forwardVector(x1)
        fxx = _finalFunc(fxx.forwardVector, x1, b1, x2, b2)
    return xv, fxx.forwardVector(xv), fxx.forwardVector

def base_curve_from_pred(pred_tensor, poly_deg=BASE_POLY_DEG, dkc=BASE_DKC):
    """Base U-net output tensor (1,3,256,256 or 3,256,256) -> callable wpc(x)."""
    transf = transforms.Compose([transforms.ToTensor(),
                                 transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    yp = pred_tensor.detach().cpu().squeeze().float()
    yR = transform_invert(yp, transf).reshape((256, 256)).astype(float)
    _, _, f1 = base_imageToLine(yR, p=poly_deg, domainCorrlation=dkc)
    def wpc(x):
        xq = np.atleast_1d(np.asarray(x, float))
        out = f1(xq.copy())
        out = np.clip(np.asarray(out, float).flatten(), 0.0, 1.0)
        return float(out[0]) if np.ndim(x) == 0 else out
    return wpc

print("✅ Cell 3 done — base-paper read-out ready (hardened)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — Our U-Net (out_channels=1) + monotone P-spline read-out  [our method]
# ═══════════════════════════════════════════════════════════════════════════
# Architecture and reader are byte-identical to our ACT-2/ACT-1 inference
# notebooks so OUR checkpoint loads and behaves exactly as in the thesis.

class OurUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_features=32, dropout=0.0):
        super().__init__(); f = init_features
        self.drop = nn.Dropout2d(p=dropout)
        self.enc1 = self._b(in_channels, f,   'enc1'); self.enc2 = self._b(f,   f*2, 'enc2')
        self.enc3 = self._b(f*2,         f*4, 'enc3'); self.enc4 = self._b(f*4, f*8, 'enc4')
        self.pool = nn.MaxPool2d(2, 2);                self.bottleneck = self._b(f*8, f*16, 'btn')
        self.up4 = nn.ConvTranspose2d(f*16, f*8, 2, 2); self.dec4 = self._b(f*16, f*8, 'dec4')
        self.up3 = nn.ConvTranspose2d(f*8,  f*4, 2, 2); self.dec3 = self._b(f*8,  f*4, 'dec3')
        self.up2 = nn.ConvTranspose2d(f*4,  f*2, 2, 2); self.dec2 = self._b(f*4,  f*2, 'dec2')
        self.up1 = nn.ConvTranspose2d(f*2,  f,   2, 2); self.dec1 = self._b(f*2,  f,   'dec1')
        self.out_conv = nn.Conv2d(f, out_channels, kernel_size=1)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(self.drop(e1)))
        e3 = self.enc3(self.pool(self.drop(e2))); e4 = self.enc4(self.pool(self.drop(e3)))
        b  = self.bottleneck(self.pool(self.drop(e4)))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], 1)); d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1)); d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out_conv(d1)
    @staticmethod
    def _b(i, ft, n):
        return nn.Sequential(OrderedDict([
            (f'{n}_c1', nn.Conv2d(i, ft, 3, padding=1, bias=False)), (f'{n}_n1', nn.BatchNorm2d(ft)), (f'{n}_r1', nn.ReLU(True)),
            (f'{n}_c2', nn.Conv2d(ft, ft, 3, padding=1, bias=False)), (f'{n}_n2', nn.BatchNorm2d(ft)), (f'{n}_r2', nn.ReLU(True))]))

# ── Our faithful monotone P-spline reader (Eilers–Marx) ──────────────────────
def _isotonic_increasing(y):
    out_v, out_w = [], []
    for v in np.asarray(y, float):
        out_v.append(v); out_w.append(1.0)
        while len(out_v) > 1 and out_v[-2] > out_v[-1]:
            v2=out_v.pop(); w2=out_w.pop(); v1=out_v.pop(); w1=out_w.pop()
            out_v.append((v1*w1+v2*w2)/(w1+w2)); out_w.append(w1+w2)
    res = []
    for v, w in zip(out_v, out_w): res.extend([v]*int(round(w)))
    res = np.array(res)
    if len(res) < len(y): res = np.append(res, [res[-1]]*(len(y)-len(res)))
    return res[:len(y)]

def _extract_curve_robust(img255):
    crop = img255[31:228, 32:230].astype(float); H, W = crop.shape
    xL = np.arange(0, 1, 1.0/W)[:W]; bg = np.percentile(crop, 95); thr = bg - 0.20*255
    y = np.full(W, np.nan)
    for i in range(W):
        col = crop[:, i]; dark = np.where(col <= thr)[0]
        if dark.size == 0: continue
        wgt = np.clip(bg - col[dark], 1e-6, None); y[i] = 1.0 - (np.sum(dark*wgt)/np.sum(wgt))/H
    m = ~np.isnan(y)
    if m.sum() < 8: return None
    xv, yv = xL[m], y[m]
    if len(yv) >= 5:
        from numpy.lib.stride_tricks import sliding_window_view
        ymed = np.median(sliding_window_view(np.pad(yv, 2, mode='edge'), 5), axis=1)
    else: ymed = yv
    ymono = _isotonic_increasing(ymed)
    return dict(xv=xv, yv=ymono, xa=float(xv.min()), xb=float(xv.max()), ytop=float(ymono.max()))

def _bspline_design(x, ik, deg=3):
    t = np.r_[[ik[0]]*deg, ik, [ik[-1]]*deg]; J = len(t)-deg-1
    B = np.zeros((len(x), J)); e = np.zeros(J)
    for j in range(J): e[:]=0.0; e[j]=1.0; B[:, j] = BSpline(t, e, deg, extrapolate=True)(x)
    return B, t, J

def _fit_monotone_pspline(xv, yv, w, n_interior=20, degree=3, lam=2.0):
    ik = np.linspace(0.0, 1.0, n_interior); B, t, J = _bspline_design(xv, ik, degree)
    sw = np.sqrt(np.asarray(w, float))[:, None]; D = np.diff(np.eye(J), n=2, axis=0); Tm = np.tril(np.ones((J, J)))
    A = (sw*B)@Tm; P = np.sqrt(lam)*(D@Tm); M = np.vstack([A, P])
    rhs = np.r_[sw[:, 0]*yv, np.zeros(P.shape[0])]; lb = np.r_[-np.inf, np.zeros(J-1)]; ub = np.full(J, np.inf)
    sol = lsq_linear(M, rhs, bounds=(lb, ub), max_iter=300, tol=1e-10)
    return t, Tm@sol.x, degree

def _build_wpc(rec, saturate=True, rated_level=1.0, flat_value_thr=0.95, lam=None):
    if lam is None: lam = WPC_LAMBDA
    xv, yv = rec['xv'], rec['yv']; xa, xb, ytop = rec['xa'], rec['xb'], rec['ytop']
    kk = max(5, len(xv)//12)
    slope_top = (yv[-1]-yv[-1-kk])/max(xv[-1]-xv[-1-kk], 1e-6) if len(xv) > kk+1 else 0.0
    max_slope = float(np.nanmax(np.gradient(yv, xv))) if len(xv) > 2 else 1.0
    climb_thr = max(0.5, 0.25*max_slope)
    mode = 'reached_rated' if ytop >= flat_value_thr else ('truncated->saturated' if slope_top > climb_thr else 'genuine_plateau')
    xf, yf, wf = list(xv), list(yv), [1.0]*len(xv); completed = False
    if saturate and mode == 'truncated->saturated':
        x_hit = min(1.0, xb + (rated_level-ytop)/max(slope_top, 1e-6))
        xs_a = np.linspace(min(1.0, xb+0.02), 1.0, 8)
        ys_a = np.maximum(np.clip(ytop+slope_top*(xs_a-xb), ytop, rated_level), np.linspace(ytop, rated_level, len(xs_a)))
        ys_a[xs_a >= x_hit] = rated_level
        xf += list(xs_a); yf += list(ys_a); wf += [0.6]*len(xs_a); completed = True
    xf = np.array(xf); yf = np.array(yf); wf = np.array(wf); o = np.argsort(xf); xf, yf, wf = xf[o], yf[o], wf[o]
    try: t, c, deg = _fit_monotone_pspline(xf, yf, wf, lam=lam); spl = BSpline(t, c, deg, extrapolate=True)
    except Exception: spl = None
    xL_, xR_ = xa, (1.0 if (saturate and completed) else xb)
    if spl is None:
        yL_ = float(yv[0]); yR_ = float(rated_level if completed else ytop)
        def wpc_fn(x):
            xq = np.atleast_1d(np.asarray(x, float)); yq = np.clip(np.interp(xq, np.r_[xa, xb], np.r_[yL_, yR_]), 0, 1)
            return float(yq[0]) if np.ndim(x) == 0 else yq
        return wpc_fn, dict(mode=mode+'|interp', completed=completed)
    yL_ = float(np.clip(spl(xL_), 0, 1)); yR_ = float(np.clip(spl(xR_), 0, 1))
    def wpc_fn(x):
        xq = np.atleast_1d(np.asarray(x, float)); y = np.empty_like(xq)
        lo = xq < xL_; hi = xq > xR_; mid = ~lo & ~hi; y[lo] = yL_; y[hi] = yR_
        if mid.any(): y[mid] = np.clip(spl(xq[mid]), 0.0, 1.0)
        return float(y[0]) if np.ndim(x) == 0 else np.clip(y, 0.0, 1.0)
    return wpc_fn, dict(mode=mode, completed=completed)

def our_curve_from_pred(pred_tensor, saturate=True):
    """Our U-net output (1,256,256) -> monotone smooth callable wpc(x)."""
    img = ((pred_tensor.squeeze().detach().cpu().float()*0.5)+0.5).clamp(0, 1).numpy()*255.0
    rec = _extract_curve_robust(img)
    if rec is None:
        return (lambda x: np.clip(np.asarray(x, float), 0, 1))
    wpc_fn, _ = _build_wpc(rec, saturate=saturate, rated_level=1.0)
    return wpc_fn

print("✅ Cell 4 done — our U-Net + P-spline reader ready")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — Shared renderer (paper Eq.24) + metrics + TP reference + S1/S2/S3
# ═══════════════════════════════════════════════════════════════════════════
# ONE renderer used by BOTH models so the input image is identical; only the
# network + its native read-out differ. This is the apples-to-apples core.

def render_scatter(ws_n, val_n):
    """(ws_n, val_n) in [0,1] -> (1,3,256,256). Marker size = base paper Eq.24."""
    n  = max(len(ws_n), 1)
    ms = float(np.clip(np.sqrt(MS_CONST / n), 0.4, 5.0))
    fig, ax = plt.subplots(figsize=(2.56, 2.56))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    ax.plot(ws_n, val_n, '.', color='black', markersize=ms)
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=100); plt.close(fig); buf.seek(0)
    pil = Image.open(buf).convert('RGB').resize((256, 256))
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)])(pil)
    return t.unsqueeze(0)

def tp_ref(ws_n, tp_n, nb=N_REF_BINS):
    """Model-INDEPENDENT scoring reference: 90th pct of TP per wind-speed bin."""
    e = np.linspace(0, 1, nb+1); xs, ys = [], []
    for i in range(nb):
        s = (ws_n >= e[i]) & (ws_n < e[i+1])
        if s.sum() >= 5:
            xs.append((e[i]+e[i+1])/2); ys.append(float(np.percentile(tp_n[s], 90)))
    return np.array(xs), np.array(ys)

def m_rmse(pred, true): return float(np.sqrt(np.mean((pred-true)**2)))
def m_mae(pred, true):  return float(np.mean(np.abs(pred-true)))
def m_mape(pred, true):
    nz = true > 0.05
    return float(100*np.mean(np.abs((pred[nz]-true[nz])/true[nz]))) if nz.sum() else np.nan
def m_wmape(pred, true):
    nz = true > 0.05
    return np.nan if nz.sum() < 3 else float(100*np.abs(pred[nz]-true[nz]).sum()/true[nz].sum())
def m_ss(pred, true, tol): return float(100*np.mean(np.abs(pred-true) <= tol))

def all_metrics(pred, true):
    return dict(RMSE=round(m_rmse(pred, true), 5), MAE=round(m_mae(pred, true), 5),
                MAPE=round(m_mape(pred, true), 2), WMAPE=round(m_wmape(pred, true), 2),
                SS_05=round(m_ss(pred, true, 0.05), 2), SS_10=round(m_ss(pred, true, 0.10), 2),
                SS_15=round(m_ss(pred, true, 0.15), 2))

def preprocess_points(ws_n, src_n, tp_n, scenario):
    """S1 raw | S2 rough curtailment clean | S3 careful clean. tp_n is guidance only."""
    if scenario == 'S1':
        return ws_n, src_n
    keep = np.ones(len(ws_n), dtype=bool)
    if scenario in ('S2', 'S3'):
        above = ws_n > 0.30; ratio = src_n/(tp_n+1e-6); keep &= ~(above & (ratio < 0.50))
    if scenario == 'S3':
        edges = np.linspace(0, 1, 26)
        for i in range(25):
            sel = keep & (ws_n >= edges[i]) & (ws_n < edges[i+1])
            if sel.sum() < 8: continue
            v = src_n[sel]; q1, q3 = np.percentile(v, 25), np.percentile(v, 75); iqr = q3-q1
            keep &= ~(sel & ((src_n < q1-1.5*iqr) | (src_n > q3+1.5*iqr)))
    if keep.sum() < 20:
        return ws_n, src_n
    return ws_n[keep], src_n[keep]

def compute_norms(df):
    """Per-turbine (WS_MAX, P_MAX) = 99.5th percentile x 1.005 (robust axis scaling)."""
    norms = {}
    for tid, g in df.groupby('Device Name'):
        ws = g['Avg Wind Speed(m/s)'].values
        tp = g[g['Avg Theoretical Active Power(kW)'] > 10]['Avg Theoretical Active Power(kW)'].values
        ws = ws[~np.isnan(ws)]; tp = tp[~np.isnan(tp)]
        if len(ws) < 100 or len(tp) < 100: continue
        norms[tid] = {'WS_MAX': max(float(np.percentile(ws, 99.5))*1.005, 1e-3),
                      'P_MAX':  max(float(np.percentile(tp, 99.5))*1.005, 1e-3)}
    return norms

# Published base-paper DITU-net (Table IV) and OUR reported results (thesis report
# Table 6). Used to auto-build comparison tables. EDIT if you update the report.
PAPER = {
 'S1': {'RMSE':0.06960,'MAE':0.04688,'MAPE':12.80,'WMAPE':13.36,'SS_05':67.91,'SS_10':86.26,'SS_15':93.91},
 'S2': {'RMSE':0.06960,'MAE':0.04684,'MAPE':12.82,'WMAPE':13.35,'SS_05':67.85,'SS_10':86.25,'SS_15':93.90},
 'S3': {'RMSE':0.06893,'MAE':0.04616,'MAPE':12.68,'WMAPE':13.14,'SS_05':68.34,'SS_10':86.50,'SS_15':93.91},
}
OURS_REPORTED = {  # from thesis results report (Table 6)
 'ACT2_2026': {  # in-distribution
   'S1': {'RMSE':0.0285,'MAE':0.0194,'MAPE':4.19,'WMAPE':3.27,'SS_05':90.31,'SS_10':98.71,'SS_15':99.79},
   'S2': {'RMSE':0.0273,'MAE':0.0185,'MAPE':3.98,'WMAPE':3.12,'SS_05':91.69,'SS_10':98.80,'SS_15':99.76},
   'S3': {'RMSE':0.0272,'MAE':0.0186,'MAPE':3.92,'WMAPE':3.13,'SS_05':91.94,'SS_10':98.71,'SS_15':99.65},
 },
 'ACT1_2025': {  # zero-shot
   'S1': {'RMSE':0.0469,'MAE':0.0327,'MAPE':9.25,'WMAPE':9.33,'SS_05':76.66,'SS_10':91.97,'SS_15':98.09},
   'S2': {'RMSE':0.0465,'MAE':0.0321,'MAPE':9.11,'WMAPE':9.15,'SS_05':77.48,'SS_10':92.23,'SS_15':98.16},
   'S3': {'RMSE':0.0467,'MAE':0.0331,'MAPE':9.89,'WMAPE':9.46,'SS_05':79.32,'SS_10':91.42,'SS_15':98.15},
 },
 # ACT2_2025: our own model numbers not in the report yet — fill when available.
}
METRIC_ORDER = ['RMSE','MAE','MAPE','WMAPE','SS_05','SS_10','SS_15']
LOWER_BETTER = {'RMSE','MAE','MAPE','WMAPE'}
print("✅ Cell 5 done — renderer, metrics, reference, scenarios, norms ready")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — Load both checkpoints (robust to {'model':sd} or raw sd) + sanity
# ═══════════════════════════════════════════════════════════════════════════
def _load_state(ckpt_path):
    obj = torch.load(ckpt_path, map_location=device)
    if isinstance(obj, dict) and 'model' in obj and isinstance(obj['model'], dict):
        return obj['model']
    if isinstance(obj, dict) and all(isinstance(v, torch.Tensor) for v in obj.values()):
        return obj
    if isinstance(obj, dict) and 'state_dict' in obj:
        return obj['state_dict']
    return obj

# Base-paper model (out_channels=3)
assert _exists(BASE_CKPT), f"Base checkpoint missing: {BASE_CKPT}"
base_model = BaseUNet(3, 3, INIT_FEATURES).to(device)
base_model.load_state_dict(_load_state(BASE_CKPT)); base_model.eval()
for p in base_model.parameters(): p.requires_grad_(False)
print(f"Base model loaded : {BASE_CKPT}")
print(f"   params={sum(p.numel() for p in base_model.parameters()):,}  (expect ~7,763,107)")

# Our model (out_channels=1) — needed for Experiment 2
our_model = None
if _exists(OUR_CKPT):
    our_model = OurUNet(3, 1, INIT_FEATURES, OUR_DROPOUT).to(device)
    our_model.load_state_dict(_load_state(OUR_CKPT)); our_model.eval()
    for p in our_model.parameters(): p.requires_grad_(False)
    print(f"Our  model loaded : {OUR_CKPT}")
    print(f"   params={sum(p.numel() for p in our_model.parameters()):,}  (expect ~7,763,041)")
else:
    print("Our model checkpoint MISSING — Experiment 2 'ours' column will be skipped.")

# sanity forward
with torch.no_grad():
    z = torch.zeros(1, 3, 256, 256, device=device)
    print("   base forward out:", tuple(base_model(z).shape))
    if our_model is not None: print("   our  forward out:", tuple(our_model(z).shape))
print("✅ Cell 6 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7 — Generic dataset loader (.csv / .xlsx), schema check, year filter
# ═══════════════════════════════════════════════════════════════════════════
REQUIRED_COLS = ['Device Name', 'time', 'Avg Wind Speed(m/s)',
                 'Avg Grid active power(kW)', 'Avg Theoretical Active Power(kW)']

def load_dataset(path, year):
    if path.lower().endswith('.csv'):
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path, engine='openpyxl')
    miss = [c for c in REQUIRED_COLS if c not in df.columns]
    assert not miss, f"{path} missing columns: {miss}"
    df['time'] = pd.to_datetime(df['time'])
    df['year_month'] = df['time'].dt.to_period('M')
    if year is not None:
        df = df[df['time'].dt.year == year].copy()
    return df

# Quick availability scan
DATASETS_OK = []
for lab, path, yr in OUR_DATASETS:
    if _exists(path):
        DATASETS_OK.append((lab, path, yr))
        print(f"   {lab:<10} {yr}: available")
    else:
        print(f"   {lab:<10} {yr}: SKIP (path not found — edit Cell 1 if you have it)")
print(f"\n{len(DATASETS_OK)} dataset(s) ready for Experiment 1")
print("✅ Cell 7 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8 — Experiment 1 core: evaluate ONE dataset with a chosen (model, reader)
# ═══════════════════════════════════════════════════════════════════════════
# reader_kind: 'base'  -> base U-net (out=3) + base imageToLine+DKC  [faithful]
#              'shared'-> base U-net (out=3) + OUR P-spline reader   [control]
# Input scatter = GP (actual grid power), exactly as deployment. Reference = TP-90th.

def eval_dataset(df, label, reader_kind):
    norms = compute_norms(df)
    months = sorted(df['year_month'].unique())
    partial = set(PARTIAL_MONTHS.get(label, []))
    rows, store = [], {}
    for scenario in ('S1', 'S2', 'S3'):
        for ym in months:
            for tid in TURBINES:
                nm = norms.get(tid)
                if nm is None: continue
                sub = df[(df['Device Name'] == tid) & (df['year_month'] == ym)]
                if len(sub) < 20: continue
                ws = sub['Avg Wind Speed(m/s)'].values
                gp = sub['Avg Grid active power(kW)'].values
                tp = sub['Avg Theoretical Active Power(kW)'].values
                ws_n = np.clip(ws/nm['WS_MAX'], 0, 1)
                gp_n = np.clip(np.maximum(gp, 0)/nm['P_MAX'], 0, 1)
                tp_n = np.clip(tp/nm['P_MAX'], 0, 1)
                v = ~(np.isnan(ws_n) | np.isnan(gp_n) | np.isnan(tp_n))
                ws_n, gp_n, tp_n = ws_n[v], gp_n[v], tp_n[v]
                if v.sum() < 20: continue
                xr, yr = tp_ref(ws_n, tp_n)
                if len(xr) < 5: continue
                ws_c, src_c = preprocess_points(ws_n, gp_n, tp_n, scenario)
                try:
                    xt = render_scatter(ws_c, src_c).to(device)
                    with torch.no_grad(): pred = base_model(xt)[0]       # (3, 256, 256)
                    if reader_kind == 'base':
                       wpc = base_curve_from_pred(pred)
                    else:   # 'shared': collapse 3-ch base output → 1-ch → our P-spline reader
                       pred_1ch = pred.mean(dim=0, keepdim=True)        # (3,256,256) → (1,256,256)
                       wpc = our_curve_from_pred(pred_1ch)
                except Exception as e:
                    continue   # graceful: drop this turbine-month, keep going
                wp = np.array([float(wpc(x)) for x in xr])
                met = all_metrics(wp, yr)
                row = dict(dataset=label, reader=reader_kind, scenario=scenario,
                           turbine=tid, month=str(ym), n=int(v.sum()),
                           is_partial=(str(ym) in partial), **met)
                rows.append(row)
                if scenario == 'S1' and reader_kind == 'base':
                    store[f"{ym}_{tid}"] = dict(fn=wpc, ws=ws_n, src=src_c, xr=xr, yr=yr, n=int(v.sum()))
    return pd.DataFrame(rows), store

print("✅ Cell 8 done — Experiment-1 evaluator ready")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 9 — Run Experiment 1: base checkpoint on OUR data (all datasets)
# ═══════════════════════════════════════════════════════════════════════════
# For every available dataset we evaluate TWO read-outs of the SAME base network:
#   'base'   = published imageToLine + DKC      (the faithful baseline)
#   'shared' = our P-spline reader on base U-net (isolates model vs. reader)
# Per-turbine-month rows are concatenated into EXP1_DF and written to disk.

EXP1_STORE = {}          # label -> S1/base store (for the WPC-grid figure)
_all_rows = []
if not DATASETS_OK:
    print("No Experiment-1 datasets available. Edit OUR_DATASETS paths in Cell 1.")
for lab, path, yr in DATASETS_OK:
    print(f"\n=== Experiment 1 :: {lab} ({yr}) ===")
    df = load_dataset(path, yr)
    print(f"   rows={len(df):,}  months={[str(m) for m in sorted(df['year_month'].unique())]}")
    for reader_kind in ('base', 'shared'):
        t0 = time.time()
        res, store = eval_dataset(df, lab, reader_kind)
        ok = len(res)
        print(f"   reader={reader_kind:<7} -> {ok:4d} turbine-month rows   ({time.time()-t0:.0f}s)")
        if ok: _all_rows.append(res)
        if reader_kind == 'base': EXP1_STORE[lab] = store
    del df; gc.collect()

EXP1_DF = pd.concat(_all_rows, ignore_index=True) if _all_rows else pd.DataFrame()
if len(EXP1_DF):
    EXP1_DF.to_csv(f"{EXP1_DIR}/exp1_per_turbine_month_ALL.csv", index=False)
    for lab in EXP1_DF['dataset'].unique():
        EXP1_DF[EXP1_DF.dataset == lab].to_csv(
            f"{EXP1_DIR}/exp1_{lab}_per_turbine_month.csv", index=False)
    print(f"\nSaved {len(EXP1_DF)} per-turbine-month rows -> {EXP1_DIR}")
print("✅ Cell 9 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 10 — Experiment-1 tables (paper-ready): summaries, comparison, reader effect
# ═══════════════════════════════════════════════════════════════════════════
HEAD = ['RMSE','MAE','MAPE (%)','WMAPE (%)','SS@.05 (%)','SS@.10 (%)','SS@.15 (%)']

def _agg(dsub):
    if len(dsub) == 0: return {m: np.nan for m in METRIC_ORDER}
    return {m: float(np.nanmean(dsub[m].values)) for m in METRIC_ORDER}

def _fmt_row(a):
    def g(k, d): return ('--' if (k not in a or pd.isna(a[k])) else f"{a[k]:.{d}f}")
    return [g('RMSE',4), g('MAE',4), g('MAPE',2), g('WMAPE',2),
            g('SS_05',2), g('SS_10',2), g('SS_15',2)]

def write_latex_table(headers, rows, path, caption, label, colspec=None):
    nc = len(headers); colspec = colspec or ('l'+'r'*(nc-1))
    def esc(s): return str(s).replace('%','\\%').replace('_','\\_').replace('&','\\&')
    L = [r'\begin{table}[!t]', r'\centering', f'\\caption{{{esc(caption)}}}',
         f'\\label{{{label}}}', f'\\begin{{tabular}}{{{colspec}}}', r'\toprule',
         ' & '.join(esc(h) for h in headers)+r' \\', r'\midrule']
    L += [' & '.join(esc(c) for c in r_)+r' \\' for r_ in rows]
    L += [r'\bottomrule', r'\end{tabular}', r'\end{table}', '']
    open(path, 'w').write('\n'.join(L))

if len(EXP1_DF) == 0:
    print("EXP1_DF empty — skipping Experiment-1 tables.")
else:
    datasets = list(EXP1_DF['dataset'].unique())
    # (A) per-dataset scenario summary (faithful base reader, complete months only)
    summ_rows_csv = []
    for lab in datasets:
        rows_disp = []
        for sc in ('S1','S2','S3'):
            sub = EXP1_DF[(EXP1_DF.dataset==lab)&(EXP1_DF.reader=='base')&
                          (EXP1_DF.scenario==sc)&(~EXP1_DF.is_partial)]
            a = _agg(sub); rows_disp.append([sc]+_fmt_row(a))
            summ_rows_csv.append(dict(dataset=lab, scenario=sc, n_turbine_months=len(sub), **a))
        write_latex_table(['Scenario']+HEAD, rows_disp,
            f"{TAB_DIR}/exp1_{lab}_scenario_summary.tex",
            f"Base-paper DITU-net evaluated on {lab} (means over turbine-months, complete months).",
            f"tab:exp1_{lab}_summary")
    pd.DataFrame(summ_rows_csv).to_csv(f"{TAB_DIR}/exp1_scenario_summary_ALL.csv", index=False)

    # (B) 3-way comparison per dataset: base-published vs base-on-ours vs ours-reported
    for lab in datasets:
        for sc in ('S1','S2','S3'):
            rows_disp, csv_rows = [], []
            rows_disp.append(['Base paper (published, their data)']+_fmt_row(PAPER[sc]))
            csv_rows.append(dict(method='base_published', scenario=sc, **PAPER[sc]))
            sub = EXP1_DF[(EXP1_DF.dataset==lab)&(EXP1_DF.reader=='base')&
                          (EXP1_DF.scenario==sc)&(~EXP1_DF.is_partial)]
            a = _agg(sub); rows_disp.append([f'Base model on {lab} (this work)']+_fmt_row(a))
            csv_rows.append(dict(method='base_on_ours', scenario=sc, **a))
            our = OURS_REPORTED.get(lab, {}).get(sc)
            if our:
                rows_disp.append([f'Ours on {lab} (reported)']+_fmt_row(our))
                csv_rows.append(dict(method='ours_reported', scenario=sc, **our))
            write_latex_table(['Method / data']+HEAD, rows_disp,
                f"{TAB_DIR}/exp1_{lab}_comparison_{sc}.tex",
                f"Apple-to-apple comparison on {lab} ({sc}): published baseline, "
                f"base checkpoint on our data, and our reported result.",
                f"tab:exp1_{lab}_cmp_{sc}")
            pd.DataFrame(csv_rows).to_csv(
                f"{TAB_DIR}/exp1_{lab}_comparison_{sc}.csv", index=False)

    # (C) reader-effect: base vs shared reader on the SAME base network
    for lab in datasets:
        rows_disp, csv_rows = [], []
        for sc in ('S1','S2','S3'):
            for rk, name in (('base','Base reader (imageToLine+DKC)'),
                             ('shared','Shared reader (our P-spline)')):
                sub = EXP1_DF[(EXP1_DF.dataset==lab)&(EXP1_DF.reader==rk)&
                              (EXP1_DF.scenario==sc)&(~EXP1_DF.is_partial)]
                a = _agg(sub); rows_disp.append([sc, name]+_fmt_row(a))
                csv_rows.append(dict(scenario=sc, reader=rk, **a))
        write_latex_table(['Scn','Read-out']+HEAD, rows_disp,
            f"{TAB_DIR}/exp1_{lab}_reader_effect.tex",
            f"Read-out ablation on {lab}: same base network, two read-outs. "
            f"Differences isolate model vs. reader.",
            f"tab:exp1_{lab}_reader", colspec='ll'+'r'*7)
        pd.DataFrame(csv_rows).to_csv(f"{TAB_DIR}/exp1_{lab}_reader_effect.csv", index=False)

    # (D) monthly summary (base reader, S1) + RMSE month x turbine pivot
    for lab in datasets:
        sub = EXP1_DF[(EXP1_DF.dataset==lab)&(EXP1_DF.reader=='base')&(EXP1_DF.scenario=='S1')]
        if len(sub)==0: continue
        mrows_disp, mrows_csv = [], []
        for ym in sorted(sub['month'].unique()):
            ms = sub[sub.month==ym]; a = _agg(ms)
            tag = ym + (' *' if ms['is_partial'].any() else '')
            mrows_disp.append([tag]+_fmt_row(a)); mrows_csv.append(dict(month=ym, **a))
        write_latex_table(['Month']+HEAD, mrows_disp,
            f"{TAB_DIR}/exp1_{lab}_monthly_S1.tex",
            f"Base-paper DITU-net on {lab}, monthly means over 20 turbines (S1). "
            f"* marks an incomplete month.", f"tab:exp1_{lab}_monthly")
        pd.DataFrame(mrows_csv).to_csv(f"{TAB_DIR}/exp1_{lab}_monthly_S1.csv", index=False)
        piv = sub.pivot_table(index='month', columns='turbine', values='RMSE', aggfunc='mean')
        piv = piv.reindex(columns=[t for t in TURBINES if t in piv.columns])
        piv.to_csv(f"{TAB_DIR}/exp1_{lab}_pivot_RMSE_S1.csv")

    print("Tables written to", TAB_DIR)
print("✅ Cell 10 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11 — Experiment-1 figures: monthly RMSE, heatmap, scenario bars, WPC grid
# ═══════════════════════════════════════════════════════════════════════════
def _save(fig, name):
    p = f"{FIG_DIR}/{name}"; fig.savefig(p, dpi=300, bbox_inches='tight'); plt.close(fig)
    return p

if len(EXP1_DF) == 0:
    print("EXP1_DF empty — skipping Experiment-1 figures.")
else:
    BASE_RMSE_REF = PAPER['S1']['RMSE']
    for lab in EXP1_DF['dataset'].unique():
        sub = EXP1_DF[(EXP1_DF.dataset==lab)&(EXP1_DF.reader=='base')&(EXP1_DF.scenario=='S1')]
        if len(sub)==0: continue
        months = sorted(sub['month'].unique())

        # (1) Monthly RMSE line with mean±std over turbines + base-paper reference
        mu = [sub[sub.month==m]['RMSE'].mean() for m in months]
        sd = [sub[sub.month==m]['RMSE'].std()  for m in months]
        fig, ax = plt.subplots(figsize=(7.2, 3.4))
        ax.axhline(BASE_RMSE_REF, ls='--', color='#d9822b',
                   label=f'Base paper S1 RMSE = {BASE_RMSE_REF:.4f}')
        ax.errorbar(range(len(months)), mu, yerr=sd, marker='o', color='#2b6cb0',
                    capsize=3, label=f'Base ckpt on {lab} (S1)')
        ax.set_xticks(range(len(months))); ax.set_xticklabels([m[-2:] for m in months])
        ax.set_xlabel('Month'); ax.set_ylabel('RMSE (TP reference)')
        ax.set_title(f'Monthly RMSE — base checkpoint on {lab} (S1, zero-shot to base model)')
        ax.legend(); ax.grid(alpha=0.3)
        _save(fig, f'exp1_{lab}_monthly_rmse.png')

        # (2) RMSE heatmap month x turbine
        piv = sub.pivot_table(index='month', columns='turbine', values='RMSE', aggfunc='mean')
        piv = piv.reindex(columns=[t for t in TURBINES if t in piv.columns])
        fig, ax = plt.subplots(figsize=(10, 0.5*len(piv)+1.5))
        im = ax.imshow(piv.values, aspect='auto', cmap='RdYlGn_r',
                       vmin=0, vmax=max(0.12, np.nanpercentile(piv.values, 95)))
        ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=90, fontsize=7)
        ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=8)
        ax.set_title(f'Base checkpoint RMSE — {lab} (S1)')
        fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label='RMSE')
        _save(fig, f'exp1_{lab}_rmse_heatmap.png')

        # (3) Scenario bars: base-on-ours vs base-paper published
        metrics = ['MAPE','WMAPE','SS_05','SS_10','SS_15']
        scn = ['S1','S2','S3']; x = np.arange(len(metrics)); w = 0.12
        fig, ax = plt.subplots(figsize=(8, 3.6))
        for j, sc in enumerate(scn):
            a = _agg(EXP1_DF[(EXP1_DF.dataset==lab)&(EXP1_DF.reader=='base')&
                             (EXP1_DF.scenario==sc)&(~EXP1_DF.is_partial)])
            ax.bar(x + (j-2.5)*w, [a[m] for m in metrics], w, label=f'{lab} {sc}')
            ax.bar(x + (j+0.5)*w, [PAPER[sc][m] for m in metrics], w,
                   hatch='//', alpha=0.55, label=f'Base paper {sc}')
        ax.set_xticks(x); ax.set_xticklabels(['MAPE','WMAPE','SS@.05','SS@.10','SS@.15'])
        ax.set_ylabel('Metric value (%)')
        ax.set_title(f'Base checkpoint on {lab} vs base paper (published)')
        ax.legend(ncol=3, fontsize=7); ax.grid(alpha=0.3, axis='y')
        _save(fig, f'exp1_{lab}_scenario_bars.png')

        # (4) 20-turbine WPC grid for a representative (most-complete) month
        store = EXP1_STORE.get(lab, {})
        if store:
            from collections import Counter
            mcount = Counter(k.split('_')[0] for k in store)
            best_month = max(mcount, key=mcount.get)
            fig, axes = plt.subplots(4, 5, figsize=(15, 11)); axes = axes.ravel()
            for k, tid in enumerate(TURBINES):
                ax = axes[k]; rec = store.get(f"{best_month}_{tid}")
                if rec is None:
                    ax.axis('off'); ax.set_title(tid, fontsize=8); continue
                ax.scatter(rec['ws'], rec['src'], s=2, c='#bbbbbb', alpha=0.4)
                ax.plot(rec['xr'], rec['yr'], 'r-', lw=1.4, label='TP ref')
                xs = np.linspace(0, 1, 100); ax.plot(xs, [rec['fn'](v) for v in xs],
                                                     'b-', lw=1.6, label='base curve')
                ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title(f"{tid} n={rec['n']}", fontsize=8)
                ax.tick_params(labelsize=6)
            axes[0].legend(fontsize=7, loc='upper left')
            fig.suptitle(f'Base checkpoint WPC — {lab} {best_month} (S1)', fontsize=13)
            fig.tight_layout(rect=[0,0,1,0.98])
            _save(fig, f'exp1_{lab}_wpc_grid_{best_month}.png')
    print("Figures written to", FIG_DIR)
print("✅ Cell 11 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 12 — Experiment 2: BOTH models on the base paper's own sample data
# ═══════════════════════════════════════════════════════════════════════════
# sample.csv has NO theoretical-power column, so we use the BASE PAPER's own
# protocol: an 80/20 split, render the 80% training scatter as the input image,
# and score each model's curve at the held-out 20% wind speeds against held-out
# ACTUAL power. Both models see the identical image and identical split.

EXP2_DF = pd.DataFrame()
EXP2_CTX = None
if not _exists(SAMPLE_CSV):
    print("SAMPLE_CSV missing — edit path in Cell 1 to run Experiment 2.")
else:
    raw = pd.read_csv(SAMPLE_CSV, header=None)   # first row IS data
    ws = pd.to_numeric(raw.iloc[:, 0], errors='coerce').values.astype(float)
    pw = pd.to_numeric(raw.iloc[:, 1], errors='coerce').values.astype(float)
    m = (~np.isnan(ws)) & (~np.isnan(pw)) & (ws >= 0)
    ws, pw = ws[m], pw[m]
    print(f"sample.csv: {len(ws):,} valid rows | WS max {ws.max():.2f} | P max {pw.max():.2f}")

    # robust per-dataset normalisation (same recipe as Experiment 1)
    WS_MAX = max(float(np.percentile(ws, 99.5))*1.005, 1e-3)
    ppos = pw[pw > 10]
    P_MAX = max(float(np.percentile(ppos if len(ppos) else pw, 99.5))*1.005, 1e-3)
    ws_n = np.clip(ws/WS_MAX, 0, 1); pw_n = np.clip(np.maximum(pw, 0)/P_MAX, 0, 1)

    rng = np.random.RandomState(SEED)
    idx = rng.permutation(len(ws_n)); ntest = int(0.2*len(idx))
    test_i, train_i = idx[:ntest], idx[ntest:]
    print(f"   split: train={len(train_i):,}  test(held-out)={len(test_i):,}")

    xt = render_scatter(ws_n[train_i], pw_n[train_i]).to(device)   # shared input image
    rows = {}; curves = {}
    with torch.no_grad(): pred_b = base_model(xt)[0]
    wpc_b = base_curve_from_pred(pred_b)
    yb = np.array([float(wpc_b(x)) for x in ws_n[test_i]])
    rows['Base paper DITU-net'] = all_metrics(yb, pw_n[test_i]); curves['Base paper DITU-net'] = wpc_b
    if our_model is not None:
        with torch.no_grad(): pred_o = our_model(xt)[0]
        wpc_o = our_curve_from_pred(pred_o)
        yo = np.array([float(wpc_o(x)) for x in ws_n[test_i]])
        rows['Our DITU-net'] = all_metrics(yo, pw_n[test_i]); curves['Our DITU-net'] = wpc_o

    EXP2_DF = pd.DataFrame([dict(model=k, **v) for k, v in rows.items()])
    EXP2_DF.to_csv(f"{EXP2_DIR}/exp2_sample_metrics.csv", index=False)
    EXP2_CTX = dict(ws_n=ws_n, pw_n=pw_n, test_i=test_i, curves=curves,
                    WS_MAX=WS_MAX, P_MAX=P_MAX)

    # paper-ready LaTeX (reuse Cell-10 helpers)
    disp = [[k]+_fmt_row(v) for k, v in rows.items()]
    write_latex_table(['Model']+HEAD, disp,
        f"{TAB_DIR}/exp2_sample_metrics.tex",
        "Both models on the base paper's own sample data (80/20 split, scored "
        "against held-out actual power; base paper protocol).", "tab:exp2_sample")

    # figure: both recovered curves over the scatter, held-out points highlighted
    fig, ax = plt.subplots(figsize=(6.4, 5.0))
    ax.scatter(ws_n[train_i], pw_n[train_i], s=3, c='#cccccc', alpha=0.5, label='train scatter')
    ax.scatter(ws_n[test_i], pw_n[test_i], s=5, c='#f0a020', alpha=0.7, label='held-out (test)')
    xs = np.linspace(0, 1, 200)
    colmap = {'Base paper DITU-net':'#c0392b', 'Our DITU-net':'#2b6cb0'}
    for name, fn in curves.items():
        ax.plot(xs, [fn(v) for v in xs], '-', lw=2, color=colmap.get(name, 'k'),
                label=f"{name} (RMSE {rows[name]['RMSE']:.4f})")
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel('Normalised wind speed'); ax.set_ylabel('Normalised power')
    ax.set_title("Experiment 2 — models on base paper sample data")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.savefig(f"{FIG_DIR}/exp2_sample_curves.png", dpi=300, bbox_inches='tight'); plt.close(fig)
    print(EXP2_DF.to_string(index=False))
    print("Experiment 2 outputs written.")
print("✅ Cell 12 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 13 — Consolidated headline cross-comparison matrix + master figure
# ═══════════════════════════════════════════════════════════════════════════
# Single matrix a reviewer can read at a glance:
#   * base paper on THEIR data (published)        * our model on OUR data (reported)
#   * base checkpoint on OUR data (Experiment 1)  * both models on THEIR data (Exp 2)

head_rows = []
def _row(cond, scope, scn, d):
    r = dict(condition=cond, data=scope, scenario=scn)
    for mk in METRIC_ORDER: r[mk] = (np.nan if d is None or mk not in d else d[mk])
    head_rows.append(r)

for sc in ('S1','S2','S3'):
    _row('Base paper (published)', 'their data', sc, PAPER[sc])
for sc in ('S1','S2','S3'):
    for lab in (['ACT2_2026','ACT1_2025','ACT2_2025'] if len(EXP1_DF) else []):
        if lab not in set(EXP1_DF['dataset']): continue
        a = _agg(EXP1_DF[(EXP1_DF.dataset==lab)&(EXP1_DF.reader=='base')&
                         (EXP1_DF.scenario==sc)&(~EXP1_DF.is_partial)])
        _row(f'Base ckpt on {lab} (Exp1)', f'our {lab}', sc, a)
for lab, sd in OURS_REPORTED.items():
    for sc, d in sd.items(): _row(f'Ours on {lab} (reported)', f'our {lab}', sc, d)
if len(EXP2_DF):
    for _, r in EXP2_DF.iterrows():
        _row(f'{r["model"]} (Exp2)', 'base sample', '80/20',
             {k: r[k] for k in METRIC_ORDER})

HEADLINE = pd.DataFrame(head_rows)
if len(HEADLINE):
    HEADLINE.to_csv(f"{TAB_DIR}/headline_cross_comparison.csv", index=False)
    disp = [[r['condition'], r['data'], r['scenario']] +
            _fmt_row({k: r[k] for k in METRIC_ORDER}) for _, r in HEADLINE.iterrows()]
    write_latex_table(['Condition','Data','Scn']+HEAD, disp,
        f"{TAB_DIR}/headline_cross_comparison.tex",
        "Consolidated cross-comparison: published baseline, base checkpoint on our "
        "data, our reported results, and both models on the base sample data.",
        "tab:headline", colspec='lll'+'r'*7)

    # master figure: RMSE across conditions (S1 for paper/Exp1/ours; Exp2 split)
    fig, ax = plt.subplots(figsize=(9, 4))
    plotd = HEADLINE[HEADLINE.scenario.isin(['S1','80/20'])].copy()
    labels = plotd['condition'].values; vals = plotd['RMSE'].values
    colors = ['#c0392b' if 'Base' in c and 'Exp1' not in c and 'Our' not in c
              else '#e08e0b' if 'Exp1' in c
              else '#2b6cb0' if 'Ours' in c or 'Our DITU' in c
              else '#777777' for c in labels]
    yy = np.arange(len(labels))
    ax.barh(yy, vals, color=colors)
    ax.set_yticks(yy); ax.set_yticklabels(labels, fontsize=8); ax.invert_yaxis()
    ax.set_xlabel('RMSE (lower is better)')
    ax.set_title('Headline RMSE across all conditions (S1 / held-out split)')
    for i, v in enumerate(vals):
        if not pd.isna(v): ax.text(v+0.001, i, f'{v:.4f}', va='center', fontsize=7)
    ax.grid(alpha=0.3, axis='x')
    fig.savefig(f"{FIG_DIR}/headline_rmse.png", dpi=300, bbox_inches='tight'); plt.close(fig)
    print(HEADLINE.to_string(index=False))
else:
    print("Nothing to consolidate yet.")
print("✅ Cell 13 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 14 — Reproducibility manifest (versions, paths, settings, definitions)
# ═══════════════════════════════════════════════════════════════════════════
import platform, scipy, torchvision, sklearn
manifest = {
  'generated_utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()),
  'python': platform.python_version(),
  'versions': {
      'torch': torch.__version__, 'torchvision': torchvision.__version__,
      'numpy': np.__version__, 'pandas': pd.__version__,
      'scipy': scipy.__version__, 'sklearn': sklearn.__version__,
  },
  'device': str(device), 'seed': SEED,
  'checkpoints': {'base': BASE_CKPT, 'ours': OUR_CKPT},
  'datasets_evaluated': [{'label': l, 'path': p, 'year': y} for (l, p, y) in DATASETS_OK],
  'sample_csv': SAMPLE_CSV,
  'evaluation': {
      'input_source': 'GP (grid active power) scatter, deployment-consistent',
      'renderer': f'shared, base-paper Eq.24 marker size = sqrt({MS_CONST}/n) clipped [0.4, 5.0]',
      'normalisation': 'per-turbine 99.5th percentile x 1.005 on WS and on TP(>10), '
                       'applied identically to model input and reference',
      'reference': f'TP 90th-percentile per wind-speed bin ({N_REF_BINS} bins, >=5 pts), '
                   'never seen by either model',
      'scenarios': {'S1': 'raw', 'S2': 'rough curtailment removal',
                    'S3': 'S2 + per-bin 1.5*IQR outlier removal'},
      'metrics': {
          'RMSE':'sqrt(mean((pred-ref)^2))', 'MAE':'mean(|pred-ref|)',
          'MAPE':'100*mean(|pred-ref|/ref) over ref>0.05',
          'WMAPE':'100*sum(|pred-ref|)/sum(ref) over ref>0.05',
          'SS@a':'100*fraction(|pred-ref|<=a), a in {0.05,0.10,0.15}'},
      'experiment2_protocol':'base-paper 80/20 split; curve scored at held-out wind '
                             'speeds vs held-out ACTUAL power (no TP column in sample.csv)',
  },
  'base_readout_fixes': [
      'imageToLine: scalar index extraction y[i]=1-where(col==vm)[0][0]/Y '
      '(original assigned a length-1 array to a scalar slot; errors on modern NumPy; '
      'numerically identical)',
      'Newton-Raphson loops bounded with guard counters + maxiter to prevent hangs on '
      'pathological curves (raises and falls back instead of looping forever)',
  ],
  'readout_modes': {'base':'published imageToLine + DKC (faithful baseline)',
                    'shared':'base U-net + our monotone P-spline reader (control)'},
}
with open(f"{OUT_ROOT}/reproducibility_manifest.json", 'w') as f:
    json.dump(manifest, f, indent=2)

# Human-readable methods note (travels inside the ZIP for the write-up)
_readme = f"""# Fair Apple-to-Apple Baseline Comparison — DITU-net

Generated: {manifest['generated_utc']} UTC

## What this package contains
Two cross-tests under ONE identical evaluation protocol, so the base-paper model
and our model are judged on the same axes, same renderer, same reference, same metrics.

### Experiment 1 — base-paper checkpoint on OUR data
The published checkpoint (`{os.path.basename(BASE_CKPT)}`) is run on each available
dataset (ACT2 2025/2026, ACT1 2025) across S1/S2/S3 and scored against the same
independent TP 90th-percentile reference used for our own results. Two read-outs are
reported: the published `imageToLine`+DKC (faithful baseline) and a shared P-spline
read-out control (isolates model vs. reader).

### Experiment 2 — both models on the base paper's OWN sample data
`sample.csv` has no theoretical-power column, so we follow the base paper's own
protocol: an 80/20 split, the 80% scatter is rendered as the input image, and each
model's curve is scored at the held-out 20% wind speeds against held-out ACTUAL power.

## Fairness controls
* One shared renderer for both models (base paper Eq.24, MS=sqrt({MS_CONST:.0f}/n)); the input
  image is identical, only the network + its native read-out differ.
* Per-turbine normalisation = 99.5th percentile x 1.005, applied identically to model
  input and reference.
* The scoring reference is never seen by either model.

## Folder map
* `tables/`  CSV + LaTeX (booktabs) — scenario summaries, 3-way comparison, reader-effect
  ablation, monthly summaries, RMSE month x turbine pivots, headline cross-comparison.
* `figures/` 300-dpi PNG — monthly RMSE vs base-paper reference, RMSE heatmaps,
  scenario bars, 20-turbine WPC grids, headline RMSE, Experiment-2 curves.
* `exp1_base_on_ours/` per-turbine-month result CSVs (Experiment 1).
* `exp2_models_on_sample/` Experiment-2 metrics.
* `reproducibility_manifest.json` exact versions, paths, settings, metric definitions.

## Notes on the base read-out (no reported value changes)
* `imageToLine` scalar-index fix (a length-1 array was assigned to a scalar slot;
  errors on modern NumPy; numerically identical).
* Newton-Raphson loops bounded with guard counters + maxiter so a pathological curve
  cannot hang the run (it raises and that turbine-month is skipped).
"""
with open(f"{OUT_ROOT}/README.md", 'w') as f:
    f.write(_readme)

print(json.dumps(manifest, indent=2))
print("✅ Cell 14 done")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 15 — Package all outputs into a single downloadable ZIP
# ═══════════════════════════════════════════════════════════════════════════
ZIP_PATH = '/kaggle/working/baseline_comparison.zip'
n_files = 0
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_ROOT):
        for fn in files:
            fp = os.path.join(root, fn)
            zf.write(fp, os.path.relpath(fp, os.path.dirname(OUT_ROOT)))
            n_files += 1
size_mb = os.path.getsize(ZIP_PATH)/1e6
print(f"Zipped {n_files} files -> {ZIP_PATH}  ({size_mb:.2f} MB)")
print("\nContents:")
for root, _, files in os.walk(OUT_ROOT):
    for fn in sorted(files):
        print("   ", os.path.relpath(os.path.join(root, fn), OUT_ROOT))
print("\nDownload baseline_comparison.zip from the Kaggle output panel (right side).")
print("✅ All done — single ZIP ready.")